<a href="https://colab.research.google.com/github/mariakellyeduarda17-ops/UFV/blob/main/TUTORIAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pysus

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 4.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.7/385.7 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.7/118.7 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.5/63.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.4/300.4 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
from pysus.online_data import SIA
import pandas as pd

def extrair_exames_acre_dez_2024():
    print("Conectando ao FTP do DATASUS (Estado: AC, Competência: 12/2024)...")

    try:
        # 1. Download dos dados de Produção Ambulatorial (PA)
        # O arquivo 'PA' contém a produção individualizada de exames
        df = SIA.download('AC', 2024, 12, group='PA')

        # 2. Filtrar apenas o Subgrupo 02 (Procedimentos Laboratoriais)
        # No SUS, exames laboratoriais clínicos começam com '0202'
        df_exames = df[df['PA_PROCID'].str.startswith('0202')].copy()

        # 3. Converter quantidades para numérico (essencial para soma)
        df_exames['PA_QTDPRO'] = pd.to_numeric(df_exames['PA_QTDPRO'])

        # 4. Mapeamento de nomes (SIGTAP simplificada para o Acre)
        # Em um cenário real, você pode baixar a tabela SIGTAP completa,
        # mas aqui usamos os exames mais frequentes no estado:
        mapeamento_sigtap = {
            '0202010476': 'Hemograma Completo',
            '0202010468': 'Dosagem de Glicose',
            '0202010310': 'Dosagem de Creatinina',
            '0202010670': 'Dosagem de Ureia',
            '0202010271': 'Dosagem de Colesterol Total',
            '0202050011': 'Urina Tipo 1 (EAS)',
            '0202080083': 'Pesquisa de Plasmodium (Malária)',
            '0202040154': 'Pesquisa de Anticorpos p/ Dengue',
            '0202010646': 'Transaminase TGO',
            '0202010654': 'Transaminase TGP',
            '0202010387': 'Dosagem de Ferritina'
        }

        # Aplicar o nome aos códigos
        df_exames['NOME_EXAME'] = df_exames['PA_PROCID'].map(mapeamento_sigtap).fillna("Outros Exames Laboratoriais")

        # 5. Agrupar e Somar as Quantidades
        resultado = df_exames.groupby(['PA_PROCID', 'NOME_EXAME'])['PA_QTDPRO'].sum().reset_index()
        resultado = resultado.sort_values(ascending=False, by='PA_QTDPRO')

        # Renomear colunas para clareza
        resultado.columns = ['Código SIGTAP', 'Nome do Exame', 'Quantidade Total']

        print("\n--- Relatório de Exames Realizados no Acre ---")
        return resultado

    except Exception as e:
        return f"Erro ao processar: {e}. (Nota: Se Dez/2024 for muito recente, o DATASUS pode ainda não ter liberado o arquivo)."

# Executar
# df_final = extrair_exames_acre_dez_2024()
# print(df_final.head(20))

In [2]:
from pysus.online_data import SIA
import pandas as pd

def relatorio_exames_acre_2024():
    # Parâmetros de busca
    estado = 'AC'
    ano = 2024
    mes = 12

    try:
        print(f"Baixando dados de {estado} para {mes}/{ano}...")
        # Baixa o grupo 'PA' (Produção Ambulatorial) que detalha os exames
        df = SIA.download(estado, ano, mes, group='PA')

        # Filtrar apenas o Grupo 02 (Procedimentos Diagnósticos)
        # e Subgrupo 02 (Laboratório Clínico)
        df_lab = df[df['PA_PROCID'].str.startswith('0202')].copy()

        # Converter quantidades para inteiro
        df_lab['PA_QTDPRO'] = pd.to_numeric(df_lab['PA_QTDPRO'])

        # Tabela De-Para (SIGTAP Simplificada) para os nomes dos exames
        nomes_exames = {
            '0202010476': 'Hemograma Completo',
            '0202010468': 'Dosagem de Glicose',
            '0202010310': 'Dosagem de Creatinina',
            '0202010670': 'Dosagem de Ureia',
            '0202080083': 'Pesquisa de Malária (Plasmodium)',
            '0202040154': 'Sorologia para Dengue',
            '0202050011': 'Exame de Urina (EAS)',
            '0202010123': 'Dosagem de Ácido Úrico',
            '0202010271': 'Colesterol Total'
        }

        # Mapear os nomes e agrupar
        df_lab['NOME_EXAME'] = df_lab['PA_PROCID'].map(nomes_exames).fillna("Outros Exames/Não Mapeados")

        final = df_lab.groupby(['PA_PROCID', 'NOME_EXAME'])['PA_QTDPRO'].sum().reset_index()
        final = final.sort_values(by='PA_QTDPRO', ascending=False)

        return final.rename(columns={'PA_PROCID': 'Código', 'PA_QTDPRO': 'Quantidade'})

    except Exception as e:
        return f"Erro: {e}. Verifique sua conexão ou a disponibilidade do arquivo no FTP do DATASUS."

# Para visualizar o resultado:
# resultado = relatorio_exames_acre_2024()
# print(resultado)

In [3]:
from pysus.online_data import SIA
import pandas as pd
import sys

def buscar_dados_acre_seguro():
    print("Tentando conexão com o DATASUS... Isso pode levar alguns minutos.")

    try:
        # Tenta baixar os dados de Produção Ambulatorial (PA)
        # O Acre (AC) é um estado leve, o arquivo deve ter cerca de 5-10MB
        df = SIA.download('AC', 2024, 12, group='PA')

        if df is None or df.empty:
            return "O servidor respondeu, mas o arquivo para AC/12/2024 parece estar vazio ou indisponível."

        print(f"Sucesso! {len(df)} registros encontrados. Processando nomes dos exames...")

        # Filtro de Laboratório (Prefixo 0202)
        df_lab = df[df['PA_PROCID'].str.startswith('0202')].copy()

        # Conversão de tipos
        df_lab['PA_QTDPRO'] = pd.to_numeric(df_lab['PA_QTDPRO'], errors='coerce').fillna(0)

        # Mapeamento essencial (Principais do Acre)
        sigtap = {
            '0202010476': 'Hemograma Completo',
            '0202010468': 'Dosagem de Glicose',
            '0202080083': 'Pesquisa de Malária (Plasmodium)',
            '0202040154': 'Sorologia para Dengue',
            '0202050011': 'Urina Tipo 1 (EAS)',
            '0202010310': 'Dosagem de Creatinina'
        }

        df_lab['NOME_EXAME'] = df_lab['PA_PROCID'].map(sigtap).fillna("Outros Exames Laboratoriais")

        # Agrupamento Final
        ranking = df_lab.groupby(['NOME_EXAME'])['PA_QTDPRO'].sum().reset_index()
        ranking = ranking.sort_values(by='PA_QTDPRO', ascending=False)

        return ranking

    except Exception as e:
        return f"Falha na conexão: {str(e)}\nDica: Verifique se você consegue acessar ftp://ftp.datasus.gov.br no seu navegador."

# Execução direta para teste
resultado = buscar_dados_acre_seguro()
print(resultado)

Tentando conexão com o DATASUS... Isso pode levar alguns minutos.
Falha na conexão: download() got an unexpected keyword argument 'group'
Dica: Verifique se você consegue acessar ftp://ftp.datasus.gov.br no seu navegador.
